# PRF Multas — notebook completo do pipeline (2007–2025)

**92.215.588 multas de trânsito da PRF**, 19 anos unificados, normalizados e tratados em uma
**base analítica de 37 colunas** (Parquet — 25 canônicas + 12 derivadas da Fase 3).

Este notebook reúne **todos os scripts usados até agora** (as células `%%writefile` recriam
os arquivos em `scripts/` com o conteúdo exato do repositório), a **implementação de colunas**,
o histórico de **alterações** e **células de validação** que rodam em segundos sobre os parquets.

## Índice
1. Configuração
2. **Fase 1 — ingestão**: `01_ingestao_brutos` valida os 19 `.csv.gz` de `dados_brutos/` (**sem Google Drive**) → `03_normalizar` (opcional/idempotente)
3. **Fase 2 — tratamento**: `04_tratar` → base analítica (28 col; **hoje 37** com as derivadas da Fase 3 — `MES` nome do mês, `ANO`, `QUARTIL` Q1–Q4 …)
4. Validações executáveis (contagens, `MES`/`QUARTIL`, tipos, nulos)
5. Histórico de alterações

> **Aviso:** as células `%%writefile` sobrescrevem `scripts/*` com o mesmo conteúdo (idempotente).
> A **validação da ingestão** (`VALIDAR_INGESTAO = True`) roda o `01_ingestao_brutos` (~1 min) a cada
> execução; as etapas pesadas (horas) ficam atrás de `EXECUTAR_PIPELINE = False`.

## 1. Configuração

Rode a partir da raiz do repo **ou** de `notebooks/` — a célula ajusta o diretório sozinha.
O único requisito é `duckdb` (`requirements.txt`: `duckdb==2.0.1`, `.venv`).

In [1]:
# Configuração: roda da raiz ou de notebooks/
from pathlib import Path
import os
import duckdb

RAIZ = Path.cwd()
if not (RAIZ / "scripts").exists() and (RAIZ.parent / "scripts").exists():
    RAIZ = RAIZ.parent
os.chdir(RAIZ)
print("RAIZ =", RAIZ)

EXECUTAR_PIPELINE = False   # True = reexecuta etapas pesadas (horas); False = só análises
VALIDAR_INGESTAO = True     # True = roda 01_ingestao_brutos (~1 min) a cada execução

con = duckdb.connect()
con.execute("SET threads=2")
con.execute("SET memory_limit='2400MB'")

RAIZ = /home/user/PRF-MULTAS-PROJETO


## 2. Fase 1 — ingestão (a partir de `dados_brutos/`, sem Google Drive)

**Fonte única:** os **19 `infracoes_<ANO>.csv.gz`** versionados nesta pasta —
mesmos arquivos do repositório. **Sem RAR, sem ZIP, sem Drive.**

Fluxo hoje: **validar/ingerir os `.csv.gz`** → (opcional) normalizar → tratar.

| Script | Etapa |
|--------|-------|
| `01_ingestao_brutos.py` | valida e ingere os 19 `.csv.gz`: contagens, header canônico, `--sha256` opcional |
| `03_normalizar.py` | normalização → 25 colunas (**opcional**: os `.gz` já estão normalizados) |
| `04_tratar.py` | Fase 2 — base analítica (28 col; hoje 37 com a Fase 3) |

> Os antigos `extract_rar.js`, `01_unificar_anos`, `01b_montar_partes` e
> `02_extrair_unificar` operavam com as fontes do Google Drive e foram
> **removidos** — a receita histórica está em `logs/ingestao_multas.md`.


### 2.1 Ingestão a partir de `dados_brutos/*.csv.gz`

`01_ingestao_brutos.py` é a porta de entrada da Fase 1: para cada ano confere
que o arquivo existe, **conta as linhas com o DuckDB lendo direto do gzip**,
confere o header canônico de 25 colunas e compara com a tabela oficial — o
**total tem que ser 92.215.588**. `--sha256` confere também os hashes do
`SHA256SUMS.txt`.

> Usa **exatamente as mesmas opções de leitura do `04`** (`strict_mode=false`,
> `quote`/`escape` explícitos) — o reader padrão do DuckDB recusa linhas de
> 2022/2024 (problema encontrado nesta varredura).


In [2]:
%%writefile scripts/01_ingestao_brutos.py
#!/usr/bin/env python3
"""
01 — Ingestão a partir de `dados_brutos/` (fonte única: `.csv.gz`, SEM Google Drive).

Os 19 CSVs anuais (2007–2025) já vivem versionados neste repositório,
compactados em gzip. Esta etapa é a porta de entrada da Fase 1 hoje: ela
**valida e ingere** direto desses arquivos — sem RAR, sem ZIP, sem Drive.

Por ano confere:
  1. o arquivo existe (`infracoes_<ANO>.csv.gz`);
  2. (opcional `--sha256`) o hash bate com `dados_brutos/SHA256SUMS.txt`;
  3. o DuckDB lê direto do gzip (`delim=';'`, `header`, `all_varchar`) e conta;
  4. a contagem bate com a tabela oficial (total 92.215.588);
  5. o cabeçalho é o schema canônico de 25 colunas, na ordem fixa.

Saída: relatório por ano (linhas, tamanho, status) + exit code 0/1.
Etapas seguintes: `03_normalizar.py` (opcional — os .gz já estão normalizados)
→ `04_tratar.py` (base analítica).

Uso:
  .venv/bin/python scripts/01_ingestao_brutos.py            # todos os 19 anos
  .venv/bin/python scripts/01_ingestao_brutos.py 2024 2025  # só os indicados
  .venv/bin/python scripts/01_ingestao_brutos.py --sha256   # + integridade dos .gz
"""
from __future__ import annotations

import hashlib
import sys
import time
from pathlib import Path

import duckdb

BASE = Path(__file__).resolve().parent.parent
BRUTOS = BASE / "dados_brutos"
SUMS = BRUTOS / "SHA256SUMS.txt"

TOTAL_ESPERADO = 92_215_588

# Tabela oficial (dados_brutos/README.md) — contagem validada por ano
ESPERADO: dict[int, int] = {
    2007: 1_855_460, 2008: 2_432_797, 2009: 3_045_607, 2010: 3_038_357,
    2011: 2_884_305, 2012: 2_843_728, 2013: 3_251_800, 2014: 3_327_193,
    2015: 5_408_986, 2016: 5_584_915, 2017: 6_023_833, 2018: 7_356_159,
    2019: 5_742_110, 2020: 5_182_537, 2021: 5_241_299, 2022: 4_135_523,
    2023: 5_946_483, 2024: 8_710_022, 2025: 10_204_474,
}

# Schema canônico de 25 colunas (ordem fixa — igual ao 03_normalizar)
CANONICA = [
    "numero_auto", "data_infracao", "hora", "tip_abordagem",
    "ind_assinou_auto", "ind_veiculo_estrangeiro", "sentido_trafego",
    "uf_placa", "uf_infracao", "num_br_infracao", "num_km_infracao",
    "nom_municipio", "cod_infracao", "descricao_abreviada",
    "enquadramento", "data_inicio_vigencia", "data_fim_vigencia",
    "med_realizada", "med_considerada", "exc_verificado", "especie",
    "nome_veiculo_marca", "nom_modelo_veiculo", "tipo_veiculo",
    "qtd_infracoes",
]


def br(n: int) -> str:
    """Número no padrão brasileiro (ponto de milhar)."""
    return f"{n:,}".replace(",", ".")


def sha256_do_arquivo(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for bloco in iter(lambda: fh.read(1 << 22), b""):
            h.update(bloco)
    return h.hexdigest()


def hashes_esperados() -> dict[str, str]:
    """Lê dados_brutos/SHA256SUMS.txt → {nome_do_arquivo: hash}."""
    if not SUMS.exists():
        raise SystemExit(f"[ERRO] {SUMS} não encontrado (necessário p/ --sha256)")
    d = {}
    for linha in SUMS.read_text(encoding="utf-8").splitlines():
        partes = linha.split()
        if len(partes) >= 2:
            d[partes[-1]] = partes[0].lower()
    return d


def validar_ano(con: duckdb.DuckDBPyConnection, ano: int,
                esperado_sha: dict[str, str] | None) -> tuple[bool, str, int]:
    """Valida um ano. Retorna (ok, motivo_do_erro_ou_vazio, linhas)."""
    arq = BRUTOS / f"infracoes_{ano}.csv.gz"
    if not arq.exists():
        return False, "arquivo ausente", 0

    if esperado_sha is not None:
        hash_esp = esperado_sha.get(arq.name)
        if hash_esp is None:
            return False, f"{arq.name} fora do SHA256SUMS.txt", 0
        if sha256_do_arquivo(arq) != hash_esp:
            return False, "sha256 divergente", 0

    # Mesmas opções do 04_tratar (strict_mode=false + quote/escape
    # explícitos) — o reader padrão recusa linhas de 2022/2024.
    q = chr(34)
    base_opts = (
        "header=true, delim=';', all_varchar=true, strict_mode=false, "
        + "quote='" + q + "', escape='" + q + "'"
    )
    cols: list[str] = []
    n = 0
    for paralelo in (True, False):
        opts = base_opts + ", parallel=" + ("true" if paralelo else "false")
        try:
            n = con.execute(
                f"SELECT COUNT(*) FROM read_csv('{arq}', {opts})"
            ).fetchone()[0]
            cols = [r[0] for r in con.execute(
                f"DESCRIBE SELECT * FROM read_csv('{arq}', {opts})"
            ).fetchall()]
            break
        except duckdb.NotImplementedException:
            continue  # reader paralelo recusou → tenta serial (caso 2023)
        except Exception as exc:  # erro real de parsing
            return False, f"leitura falhou: {str(exc)[:80]}", 0
    else:
        return False, "leitura falhou (paralelo e serial)", 0

    if n != ESPERADO[ano]:
        return False, f"{br(n)} linhas != {br(ESPERADO[ano])} esperadas", n
    if cols != CANONICA:
        return False, f"header != canônico ({len(cols)} col vs 25)", n
    return True, "", n


def main(argv: list[str]) -> int:
    anos = [int(a) for a in argv if a.isdigit()]
    checar_sha = "--sha256" in argv
    if not anos:
        anos = sorted(ESPERADO)

    t0 = time.time()
    con = duckdb.connect(config={"threads": "2", "memory_limit": "1500MB"})
    esperado_sha = hashes_esperados() if checar_sha else None

    print("=" * 78)
    print("INGESTÃO — fonte: dados_brutos/*.csv.gz (sem Google Drive)")
    print(f"  anos: {min(anos)}–{max(anos)} · sha256: {'sim' if checar_sha else 'não'}")
    print("=" * 78)
    print(f"  {'ANO':>4}  {'TAMANHO':>10}  {'LINHAS':>13}  {'ESPERADO':>13}  STATUS")

    falhas: list[str] = []
    total = 0
    for ano in anos:
        arq = BRUTOS / f"infracoes_{ano}.csv.gz"
        ok, motivo, n = validar_ano(con, ano, esperado_sha)
        total += n
        tam = f"{arq.stat().st_size / 1e6:.1f} MB" if arq.exists() else "—"
        status = "OK" if ok else f"FALHA — {motivo}"
        print(f"  {ano:>4}  {tam:>10}  {br(n):>13}  "
              f"{br(ESPERADO[ano]):>13}  {status}")
        if not ok:
            falhas.append(f"{ano}: {motivo}")

    print("-" * 78)
    if len(anos) == len(ESPERADO):
        print(f"  {'TOTAL':>4}  {'':>10}  {br(total):>13}  "
              f"{br(TOTAL_ESPERADO):>13}  {'OK' if total == TOTAL_ESPERADO else 'FALHA'}")
        if total != TOTAL_ESPERADO:
            falhas.append(f"total {br(total)} != {br(TOTAL_ESPERADO)}")

    if falhas:
        print(f"\n[ERRO] ingestão com {len(falhas)} falha(s):")
        for f in falhas:
            print(f"  - {f}")
        return 1

    print(f"\nINGESTAO_OK — {len(anos)}/19 anos validados a partir de "
          f"dados_brutos/*.csv.gz ({time.time() - t0:.0f}s, sem Google Drive)")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))


Overwriting scripts/01_ingestao_brutos.py


In [3]:
if VALIDAR_INGESTAO:
    import subprocess, sys
    subprocess.run([sys.executable, "scripts/01_ingestao_brutos.py"], check=True)
else:
    print("VALIDAR_INGESTAO=False — pulando validação da ingestão")

INGESTÃO — fonte: dados_brutos/*.csv.gz (sem Google Drive)
  anos: 2007–2025 · sha256: não
   ANO     TAMANHO         LINHAS       ESPERADO  STATUS
  2007     30.5 MB      1.855.460      1.855.460  OK
  2008     43.7 MB      2.432.797      2.432.797  OK
  2009     57.8 MB      3.045.607      3.045.607  OK
  2010     57.6 MB      3.038.357      3.038.357  OK
  2011     55.1 MB      2.884.305      2.884.305  OK
  2012     54.1 MB      2.843.728      2.843.728  OK
  2013     61.7 MB      3.251.800      3.251.800  OK
  2014     61.4 MB      3.327.193      3.327.193  OK
  2015    115.9 MB      5.408.986      5.408.986  OK
  2016    123.1 MB      5.584.915      5.584.915  OK
  2017    132.0 MB      6.023.833      6.023.833  OK
  2018    162.0 MB      7.356.159      7.356.159  OK
  2019    137.1 MB      5.742.110      5.742.110  OK
  2020    120.6 MB      5.182.537      5.182.537  OK
  2021    123.0 MB      5.241.299      5.241.299  OK
  2022    147.5 MB      4.135.523      4.135.523  OK
  20

### 2.5 Normalização → schema canônico de 25 colunas (`03`) — opcional

> **Hoje não é necessária:** os `.csv.gz` de `dados_brutos/` já estão
> normalizados (é o estado versionado). Rodar somente ao regenerar a partir
> de fontes brutas — receita histórica em `logs/ingestao_multas.md`.

Aplica em 19/19 anos:
- **mapeamento de nomes** (NFKD-fold) das três eras → 25 colunas na ordem fixa;
- `SELECT` canônico com `all_varchar` → TRIM, tokens (`NA/N-I/N-A`/vazio) → `NULL`,
  datas → `AAAA-MM-DD` (defensivo `DD/MM`), `hora` → `HH:MM:SS`;
- validação: contagem entrada == saída, header md5 único (`8fe1a7bb…`);
- **fallback automático `parallel=false`** quando o reader paralelo do DuckDB recusa o
  arquivo (caso `infracoes_2023.csv.gz`) — leitura serial dentro do limite de memória;
- grava `.csv` (depois `gzip -6` → `.csv.gz`) com `os.replace` atômico.

**Total fase 1: 92.215.588 linhas · 19 anos · header idêntico.**

In [4]:
%%writefile scripts/03_normalizar.py
#!/usr/bin/env python3
"""
Etapa 03 — Normalização dos CSVs unificados de `dados_brutos`.

Padroniza TODOS os anos (2007–2025) para um schema canônico único:

  * 25 colunas canônicas (união dos schemas das três eras), ordem fixa,
    nomes em snake_case
  * Delimitador único `;`, UTF-8, LF, sem BOM, aspas mínimas
  * Datas em AAAA-MM-DD (converte DD/MM/AAAA se aparecer; já é o padrão)
  * `hora` em HH:MM:SS (anos que só tinham o dígito da hora → 'HH:00:00')
  * TRIM em todos os campos
  * Tokens de ausência ('NA', 'N/I', 'N/A' e vazio) → vazio (NULL)

Validação por ano (obrigatória): contagem de entrada == contagem de saída,
`data_infracao` 100% ISO e `hora` 100% HH:MM:SS no arquivo normalizado.
Só depois disso o temporário substitui o arquivo final.

Aceita fonte `.csv` ou `.csv.gz` (a etapa de compactação prévia libera disco).
Saída: sobrescreve `dados_brutos/infracoes_<ANO>.csv` (via `.norm.csv` + replace).

Uso:
  .venv/bin/python scripts/03_normalizar.py              # todos os anos
  .venv/bin/python scripts/03_normalizar.py 2007 2008    # só os indicados
"""
from __future__ import annotations

import csv
import gzip
import os
import shutil
import sys
import time
import unicodedata
from pathlib import Path

import duckdb

BASE = Path(__file__).resolve().parent.parent
BRUTOS = BASE / "dados_brutos"

# ---------------------------------------------------------------------------
# Schema canônico (união de todas as eras, ordem fixa)
# ---------------------------------------------------------------------------
CANONICA = [
    "numero_auto",           # era2/3 (2019+)            — era1: ausente
    "data_infracao",         # era1: dat_infracao
    "hora",                  # era1 2015+                — 2007–2014: ausente
    "tip_abordagem",
    "ind_assinou_auto",
    "ind_veiculo_estrangeiro",
    "sentido_trafego",       # era1: ind_sentido_trafego
    "uf_placa",              # 2021: ausente
    "uf_infracao",
    "num_br_infracao",
    "num_km_infracao",
    "nom_municipio",
    "cod_infracao",          # era1 2017+                — 2007–2016: ausente
    "descricao_abreviada",
    "enquadramento",
    "data_inicio_vigencia",
    "data_fim_vigencia",
    "med_realizada",         # era2/3: Medição Infração
    "med_considerada",
    "exc_verificado",
    "especie",
    "nome_veiculo_marca",
    "nom_modelo_veiculo",    # era2 2019–2021: ausente
    "tipo_veiculo",          # só era3 (2022+)
    "qtd_infracoes",         # era2 2020+ / era3
]

DATAS = {"data_infracao", "data_inicio_vigencia", "data_fim_vigencia"}
TOKENS_AUSENCIA = ("", "NA", "N/I", "N/A")


def _n(nome: str) -> str:
    """Normaliza nome de coluna: sem acento, minúsculo, sem espaços extras."""
    s = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return " ".join(s.lower().split())


# Mapeia nome-de-coluna-normalizado -> coluna canônica (inclui identidade p/ re-execução)
MAPEAMENTO: dict[str, str] = {c: c for c in CANONICA}
MAPEAMENTO.update({
    # era 1 (2007–2018, snake_case)
    "dat_infracao": "data_infracao",
    "ind_sentido_trafego": "sentido_trafego",
    # era 2/3 (2019–2025, pt-BR)
    "numero do auto": "numero_auto",
    "data da infracao (dd/mm/aaaa)": "data_infracao",
    "indicador de abordagem": "tip_abordagem",
    "assinatura do auto": "ind_assinou_auto",
    "indicador veiculo estrangeiro": "ind_veiculo_estrangeiro",
    "sentido trafego": "sentido_trafego",
    "uf placa": "uf_placa",
    "uf infracao": "uf_infracao",
    "br infracao": "num_br_infracao",
    "km infracao": "num_km_infracao",
    "municipio": "nom_municipio",
    "codigo da infracao": "cod_infracao",
    "descricao abreviada infracao": "descricao_abreviada",
    "enquadramento da infracao": "enquadramento",
    "inicio vigencia da infracao": "data_inicio_vigencia",
    "fim vigencia infracao": "data_fim_vigencia",
    "medicao infracao": "med_realizada",
    "medicao considerada": "med_considerada",
    "excesso verificado": "exc_verificado",
    "descricao especie veiculo": "especie",
    "descricao marca veiculo": "nome_veiculo_marca",
    "descricao modelo veiculo": "nom_modelo_veiculo",
    "descricao tipo veiculo": "tipo_veiculo",
    "hora infracao": "hora",
    "qtd infracoes": "qtd_infracoes",
})


def _q(ident: str) -> str:
    return '"' + ident.replace('"', '""') + '"'


def _expr_data(q: str) -> str:
    in_list = ", ".join(f"'{t}'" for t in TOKENS_AUSENCIA)
    return (
        f"CASE WHEN trim({q}) IN ({in_list}) THEN NULL "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$') THEN trim({q}) "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{4}}$') "
        f"THEN substr(trim({q}), 7, 4) || '-' || substr(trim({q}), 4, 2) || '-' || substr(trim({q}), 1, 2) "
        f"ELSE trim({q}) END"
    )


def _expr_hora(q: str) -> str:
    in_list = ", ".join(f"'{t}'" for t in TOKENS_AUSENCIA)
    return (
        f"CASE WHEN trim({q}) IN ({in_list}) THEN NULL "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{2}}:[0-9]{{2}}:[0-9]{{2}}$') THEN trim({q}) "
        f"WHEN regexp_matches(trim({q}), '^[0-9]{{1,2}}$') "
        f"AND try_cast(trim({q}) AS INTEGER) BETWEEN 0 AND 23 "
        f"THEN lpad(trim({q}), 2, '0') || ':00:00' "
        f"ELSE trim({q}) END"
    )


def _expr_comum(q: str) -> str:
    in_list = ", ".join(f"'{t}'" for t in TOKENS_AUSENCIA)
    return f"CASE WHEN trim({q}) IN ({in_list}) THEN NULL ELSE trim({q}) END"


def _fonte(ano: int) -> Path:
    gz = BRUTOS / f"infracoes_{ano}.csv.gz"
    csv_p = BRUTOS / f"infracoes_{ano}.csv"
    if gz.exists():
        return gz
    if csv_p.exists():
        return csv_p
    raise SystemExit(f"[ERRO] fonte não encontrada para {ano} ({csv_p.name} / {gz.name})")


def _ler_cabecalho(src: Path) -> tuple[list[str], str]:
    opener = gzip.open if src.suffix == ".gz" else open
    with opener(src, "rb") as fh:
        cab = fh.readline().decode("utf-8", "replace").rstrip("\r\n")
    # delim detectado pelo próprio cabeçalho (funciona também p/ re-execução)
    delim = ";" if ";" in cab else ","
    cols = next(csv.reader([cab], delimiter=delim))
    return cols, delim


def normalizar_ano(ano: int) -> tuple[int, float]:
    src = _fonte(ano)
    dst = BRUTOS / f"infracoes_{ano}.csv"
    tmp = dst.with_name(f"infracoes_{ano}.norm.csv")
    if tmp.exists():
        tmp.unlink()

    cols, delim = _ler_cabecalho(src)

    # ---- monta o SELECT no schema canônico -------------------------------
    origem: dict[str, str] = {}
    desconhecidas: list[str] = []
    for c in cols:
        chave = _n(c)
        if chave not in MAPEAMENTO:
            desconhecidas.append(c)
            continue
        alvo = MAPEAMENTO[chave]
        if alvo in origem:
            raise SystemExit(
                f"[ERRO] {ano}: duas colunas de origem mapeiam para '{alvo}': "
                f"'{origem[alvo]}' e '{c}'"
            )
        origem[alvo] = c
    if desconhecidas:
        raise SystemExit(f"[ERRO] {ano}: colunas de origem sem mapeamento: {desconhecidas}")

    partes: list[str] = []
    for alvo in CANONICA:
        if alvo not in origem:
            partes.append(f'CAST(NULL AS VARCHAR) AS "{alvo}"')
            continue
        q = _q(origem[alvo])
        if alvo in DATAS:
            e = _expr_data(q)
        elif alvo == "hora":
            e = _expr_hora(q)
        else:
            e = _expr_comum(q)
        partes.append(f'{e} AS "{alvo}"')
    selecao = ", ".join(partes)

    con = duckdb.connect()
    # tuning p/ máquina pequena (2 cores / 3,8 GB): 1 thread + sem buffer de
    # ordenação (com 1 thread a ordem de origem é preservada) + spill em disco
    (BRUTOS / "_ddb_tmp").mkdir(exist_ok=True)
    con.execute("SET threads=1")
    con.execute("SET preserve_insertion_order=false")
    con.execute("SET memory_limit='2400MB'")
    con.execute(f"SET temp_directory='{BRUTOS / '_ddb_tmp'}'")
    caminho = str(src.resolve())
    leitura = (
        f"read_csv('{caminho}', header=true, delim='{delim}', quote='\"', escape='\"', "
        "all_varchar=true, encoding='utf-8', strict_mode=false, parallel=true)"
    )

    t0 = time.time()
    try:
        n_in = con.execute(f"SELECT count(*) FROM {leitura}").fetchone()[0]
    except duckdb.NotImplementedException:
        # alguns arquivos nao suportam o CSV reader paralelo ("full read");
        # cai para leitura serial (bufferiza o descompactado — só viável com
        # arquivos < limite de memória do DuckDB)
        leitura = (
            f"read_csv('{caminho}', header=true, delim='{delim}', quote='\"', escape='\"', "
            "all_varchar=true, encoding='utf-8', strict_mode=false, parallel=false)"
        )
        print(f"  [INFO {ano}] reader paralelo indisponível — usando leitura serial")
        n_in = con.execute(f"SELECT count(*) FROM {leitura}").fetchone()[0]

    con.execute(
        f"COPY (SELECT {selecao} FROM {leitura}) TO '{tmp.resolve()}' "
        "(HEADER true, DELIMITER ';', QUOTE '\"', ESCAPE '\"', NULL '')"
    )

    leitura_tmp = (
        f"read_csv('{tmp.resolve()}', header=true, delim=';', quote='\"', escape='\"', "
        "all_varchar=true, encoding='utf-8', strict_mode=false, parallel=true)"
    )
    n_out, bad_data, bad_hora = con.execute(
        f"SELECT count(*), "
        f"coalesce(sum(CASE WHEN data_infracao IS NOT NULL AND data_infracao <> '' "
        f"AND NOT regexp_matches(data_infracao, '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$') "
        f"THEN 1 ELSE 0 END), 0), "
        f"coalesce(sum(CASE WHEN hora IS NOT NULL AND hora <> '' "
        f"AND NOT regexp_matches(hora, '^[0-9]{{2}}:[0-9]{{2}}:[0-9]{{2}}$') "
        f"THEN 1 ELSE 0 END), 0) "
        f"FROM {leitura_tmp}"
    ).fetchone()
    con.close()

    if n_in != n_out:
        tmp.unlink(missing_ok=True)
        raise SystemExit(f"[ERRO] {ano}: contagem divergente — entrada {n_in:,} vs saída {n_out:,}")
    if bad_data or bad_hora:
        tmp.unlink(missing_ok=True)
        raise SystemExit(
            f"[ERRO] {ano}: valores fora do padrão no normalizado — "
            f"data_infracao={bad_data}, hora={bad_hora}"
        )

    # ---- troca definitiva -----------------------------------------------
    os.replace(tmp, dst)
    if src != dst:            # remove o .gz depois da troca validada
        src.unlink()

    dt = time.time() - t0
    mb = dst.stat().st_size / 1e6
    ausentes = [c for c in CANONICA if c not in origem]
    info = f" | ausentes(vazio): {', '.join(ausentes)}" if ausentes else ""
    print(
        f"OK {ano}: {n_out:,} registros | 25 colunas | delim=';' | "
        f"{dst.name} ({mb:,.1f} MB, {dt:.1f}s){info}",
        flush=True,
    )
    livre = shutil.disk_usage(BRUTOS).free / 1e9
    print(f"     disco livre: {livre:.2f} GB", flush=True)
    return n_out, dt


def main(argv: list[str]) -> int:
    if argv:
        anos = [int(a) for a in argv]
    else:
        anos = sorted(
            int(p.stem.split("_")[1].split(".")[0])
            for p in BRUTOS.glob("infracoes_*.csv*")
            if not p.name.endswith(".norm.csv")
        )
    total = 0
    t0 = time.time()
    for ano in anos:
        n, _ = normalizar_ano(ano)
        total += n
    print(f"\n=== RESUMO NORMALIZAÇÃO ===")
    print(f"  anos: {min(anos)}–{max(anos)} ({len(anos)} arquivos)")
    print(f"  registros: {total:,}")
    print(f"  tempo total: {time.time() - t0:.1f}s")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))

Overwriting scripts/03_normalizar.py


## 3. Fase 2 — tratamento e implementação de colunas (`04_tratar.py`)

Le os 19 `.csv.gz` (1 passada de gzip/ano em tabela TEMP `stage`; fallback serial no 2023)
e grava **`dados_tratados/base_analitica/infracoes_YYYY.parquet`** (ZSTD, **1,4 GB**).

### 3.1 Padronização (25 colunas de origem → todas MAIÚSCULAS)

| Regra | Detalhe |
|-------|---------|
| Sentinela → `NULL` | `'-1`, `-1`, `NA`, `N/A`, `N/I`, `N-I`, `N-A`, `NULL`, `NULO`, `#N/A` |
| Texto | `TRIM` + **MAIÚSCULAS** + acentos→ASCII (`TRAÇÃO`=`TRACAO`; `SÃO PAULO`=`SAO PAULO`) |
| `DATA_*` | `DATE` (ISO ou `DD/MM/AAAA` defensivo; inválida → `NULL`) |
| `HORA` | `TIME` |
| `NUM_KM_INFRACAO` | `DOUBLE` (aceita vírgula; negativo → `NULL`) |
| `QTD_INFRACOES` / `NUM_BR_INFRACAO` | `BIGINT` (`<=0` / não numérico → `NULL`) |
| `UF_PLACA`/`UF_INFRACAO` | só 27 UFs válidas → resto `NULL` (3.941.160 células) |
| `COD_INFRACAO` | mantido como **código texto** |

### 3.2 Colunas derivadas (25 → **28**)

- **`MES`** (`VARCHAR`): **nome do mês** — `JANEIRO…DEZEMBRO` (acento→ASCII: `MARCO`).
  *(alteração: era SMALLINT 1–12; trocado a pedido)*
- **`ANO`** (`SMALLINT`): ano de `DATA_INFRACAO`.
- **`QUARTIL`** (`VARCHAR`): **4 períodos de 3 meses** — `Q1`=jan-mar, `Q2`=abr-jun,
  `Q3`=jul-set, `Q4`=out-dez *(pedido: "a cada 4 meses" com Q1–Q4 = 4×3 meses)*.

**Nenhuma linha removida: 92.215.588 entram e saem.**

In [5]:
%%writefile scripts/04_tratar.py
#!/usr/bin/env python3
"""
04_tratar.py — Fase 2: tratamento e padronizacao -> BASE ANALITICA (Parquet).

Le os 19 CSV normalizados (dados_brutos/infracoes_*.csv.gz) e gera:
  dados_tratados/base_analitica/infracoes_YYYY.parquet   (28 colunas, ZSTD)
  dados_tratados/resumo_colunas.md                       (estatisticas + resumo)

Padronizacao (todas as colunas de texto):
  sentinela ('-1, -1, NA, N/A, N/I, N-I, N-A, NULL, NULO...) -> NULL
  senao: TRIM -> UPPER -> acentos -> ASCII -> '' -> NULL
Tipos:
  DATA_*        DATE  (ISO 'AAAA-MM-DD' ou 'DD/MM/AAAA' defensivo; invalido -> NULL)
  HORA          TIME  (invalido -> NULL)
  NUM_KM        DOUBLE (virgula ou ponto aceito; negativo -> NULL)
  QTD_INFRACOES BIGINT (<= 0 -> NULL)
  NUM_BR        BIGINT (nao numerico -> NULL)
  COD_INFRACAO  VARCHAR (e' codigo, nao quantidade)
  UF_*          VARCHAR (apenas 27 UFs validas; resto -> NULL)
Derivadas (a partir de DATA_INFRACAO):
  MES     VARCHAR nome do mes (JANEIRO..DEZEMBRO; acento->ASCII: MARCO)
  ANO     SMALLINT (ano da infracao)
  QUARTIL Q1=jan-mar | Q2=abr-jun | Q3=jul-set | Q4=out-dez
Linhas: NENHUMA removida (92.215.588 entram e saem).

Uso: .venv/bin/python scripts/04_tratar.py [ANO ...] [--resumo]
"""
from __future__ import annotations

import json
import sys
import time
import unicodedata
from pathlib import Path

import duckdb

RAIZ = Path(__file__).resolve().parent.parent
BRUTOS = RAIZ / "dados_brutos"
TRATADOS = RAIZ / "dados_tratados"
BASE_DIR = TRATADOS / "base_analitica"
TMP_DIR = TRATADOS / "_ddb_tmp"
VALIDACAO = TRATADOS / "_validacao.jsonl"

TOTAL_ESPERADO = 92_215_588

# ---------------------------------------------------------------------------
# mapeamento de acentos (construido com unicodedata para garantir paridade)
# ---------------------------------------------------------------------------
_par: dict[str, str] = {}
for _ch in "ÁÀÂÃÄÅÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑáàâãäåéèêëíìîïóòôõöúùûüçñ":
    _base = unicodedata.normalize("NFD", _ch)[0]
    _par[_ch] = _base
ACC_SRC = "".join(_par.keys())
ACC_DST = "".join(_par.values())

SENTINELAS = ("-1", "'-1", "NA", "N/A", "N/I", "N-I", "N-A", "NULL", "NULO", "#N/A")
SENT_SQL = ", ".join("'" + s.replace("'", "''") + "'" for s in SENTINELAS)

UFS = (
    "AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO"
).split()
UFS_SQL = ", ".join("'" + u + "'" for u in UFS)

UFS_NAO_NUM = f"({UFS_SQL})"


def p(col: str) -> str:
    """Padroniza texto: sentinela -> NULL; senao TRIM+UPPER+dobrar acentos; '' -> NULL."""
    v = f"upper(trim({col}))"
    return (
        f"CASE WHEN {col} IS NULL OR {v} IN ({SENT_SQL}) THEN NULL "
        f"ELSE NULLIF(translate({v}, '{ACC_SRC}', '{ACC_DST}'), '') END"
    )


def ufs(col: str) -> str:
    """UF: apenas siglas validas (27 UFs); sentinela/lixo -> NULL."""
    v = f"upper(trim({col}))"
    return f"CASE WHEN {v} IN {UFS_NAO_NUM} THEN {v} END"


def dt(col: str) -> str:
    """Data: ISO ou DD/MM/AAAA defensivo; invalida -> NULL."""
    return (
        f"CASE WHEN {col} IS NULL THEN NULL "
        f"WHEN trim({col}) ~ '^[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}$' "
        f"THEN try_cast(trim({col}) AS DATE) "
        f"WHEN trim({col}) ~ '^[0-9]{{2}}/[0-9]{{2}}/[0-9]{{4}}$' "
        f"THEN try_strptime(trim({col}), '%d/%m/%Y')::DATE "
        f"ELSE NULL END"
    )


HORA = "try_cast(trim(hora) AS TIME)"
KM = "try_cast(nullif(replace(trim(num_km_infracao), ',', '.'), '') AS DOUBLE)"
KM_OK = f"CASE WHEN {KM} IS NULL OR {KM} < 0 THEN NULL ELSE {KM} END"
QTD = "try_cast(trim(qtd_infracoes) AS BIGINT)"
QTD_OK = f"CASE WHEN {QTD} IS NULL OR {QTD} <= 0 THEN NULL ELSE {QTD} END"
BR = "try_cast(trim(num_br_infracao) AS BIGINT)"

COLS_SELECT = f"""
    {p('numero_auto')}                          AS NUMERO_AUTO,
    {dt('data_infracao')}                       AS DATA_INFRACAO,
    {HORA}                                      AS HORA,
    {p('tip_abordagem')}                        AS TIP_ABORDAGEM,
    {p('ind_assinou_auto')}                     AS IND_ASSINOU_AUTO,
    {p('ind_veiculo_estrangeiro')}              AS IND_VEICULO_ESTRANGEIRO,
    {p('sentido_trafego')}                      AS SENTIDO_TRAFEGO,
    {ufs('uf_placa')}                           AS UF_PLACA,
    {ufs('uf_infracao')}                        AS UF_INFRACAO,
    {BR}                                        AS NUM_BR_INFRACAO,
    {KM_OK}                                     AS NUM_KM_INFRACAO,
    {p('nom_municipio')}                        AS NOM_MUNICIPIO,
    {p('cod_infracao')}                         AS COD_INFRACAO,
    {p('descricao_abreviada')}                  AS DESCRICAO_ABRVIADA,
    {p('enquadramento')}                        AS ENQUADRAMENTO,
    {dt('data_inicio_vigencia')}                AS DATA_INICIO_VIGENCIA,
    {dt('data_fim_vigencia')}                   AS DATA_FIM_VIGENCIA,
    {p('med_realizada')}                        AS MED_REALIZADA,
    {p('med_considerada')}                      AS MED_CONSIDERADA,
    {p('exc_verificado')}                       AS EXC_VERIFICADO,
    {p('especie')}                              AS ESPECIE,
    {p('nome_veiculo_marca')}                   AS NOME_VEICULO_MARCA,
    {p('nom_modelo_veiculo')}                   AS NOM_MODELO_VEICULO,
    {p('tipo_veiculo')}                         AS TIPO_VEICULO,
    {QTD_OK}                                    AS QTD_INFRACOES
"""

SQL_TRATAR = f"""
SELECT
    {COLS_SELECT},
    CASE MES_NUM
        WHEN 1  THEN 'JANEIRO'
        WHEN 2  THEN 'FEVEREIRO'
        WHEN 3  THEN 'MARCO'
        WHEN 4  THEN 'ABRIL'
        WHEN 5  THEN 'MAIO'
        WHEN 6  THEN 'JUNHO'
        WHEN 7  THEN 'JULHO'
        WHEN 8  THEN 'AGOSTO'
        WHEN 9  THEN 'SETEMBRO'
        WHEN 10 THEN 'OUTUBRO'
        WHEN 11 THEN 'NOVEMBRO'
        WHEN 12 THEN 'DEZEMBRO'
    END AS MES,
    ANO,
    CASE
        WHEN MES_NUM IS NULL THEN NULL
        WHEN MES_NUM <= 3  THEN 'Q1'
        WHEN MES_NUM <= 6  THEN 'Q2'
        WHEN MES_NUM <= 9  THEN 'Q3'
        ELSE 'Q4'
    END AS QUARTIL
FROM (
    SELECT
        *,
        EXTRACT(MONTH FROM {dt('data_infracao')})::SMALLINT AS MES_NUM,
        EXTRACT(YEAR  FROM {dt('data_infracao')})::SMALLINT AS ANO
    FROM stage
)
"""

SQL_VALIDACAO = f"""
SELECT
    count(*)                                                          AS linhas,
    count(*) FILTER (WHERE data_infracao IS NOT NULL AND _d  IS NULL) AS data_ruim,
    count(*) FILTER (WHERE data_infracao IS NOT NULL AND _d  IS NOT NULL
                      AND EXTRACT(YEAR FROM _d)::INT <> {{ANO}})       AS ano_divergente,
    count(*) FILTER (WHERE hora IS NOT NULL AND _h IS NULL)           AS hora_ruima,
    count(*) FILTER (WHERE uf_placa IS NOT NULL AND _ufp IS NULL)     AS uf_placa_invalida,
    count(*) FILTER (WHERE uf_infracao IS NOT NULL AND _ufi IS NULL)  AS uf_infracao_invalida,
    count(*) FILTER (WHERE num_km_infracao IS NOT NULL AND _km IS NULL) AS km_invalido,
    count(*) FILTER (WHERE qtd_infracoes IS NOT NULL AND _q IS NULL)  AS qtd_invalida,
    count(*) FILTER (WHERE num_br_infracao IS NOT NULL AND _br IS NULL) AS br_invalido,
    count(*) FILTER (WHERE upper(trim(especie)) IN ({SENT_SQL})) AS especie_sentinela
FROM (
    SELECT
        data_infracao, hora, uf_placa, uf_infracao,
        num_km_infracao, qtd_infracoes, num_br_infracao, especie,
        {dt('data_infracao')} AS _d,
        {HORA}                AS _h,
        {ufs('uf_placa')}     AS _ufp,
        {ufs('uf_infracao')}  AS _ufi,
        {KM_OK}               AS _km,
        {QTD_OK}              AS _q,
        {BR}                  AS _br
    FROM stage
)
"""


def criar_stage(con: duckdb.DuckDBPyConnection, arquivo: Path, paralelo: bool = True) -> None:
    """Materializa o CSV (1x decompress) em TEMP; metricas e COPY leem daqui."""
    q = chr(34)  # aspas duplas dentro do SQL: quote='"'
    opts = (
        "header=true, delim=';', all_varchar=true, strict_mode=false, "
        + "quote='" + q + "', escape='" + q + "', "
        + "parallel=" + ("true" if paralelo else "false")
    )
    con.execute("DROP TABLE IF EXISTS stage")
    con.execute(
        f"CREATE TEMP TABLE stage AS "
        f"SELECT * FROM read_csv('{arquivo}', {opts})"
    )


def tratar_ano(con: duckdb.DuckDBPyConnection, ano: int) -> dict:
    arquivo = BRUTOS / f"infracoes_{ano}.csv.gz"
    if not arquivo.exists():
        raise FileNotFoundError(
            f"{arquivo} nao encontrado (remonte as partes: cat {arquivo.name}.* > {arquivo.name})"
        )
    destino = BASE_DIR / f"infracoes_{ano}.parquet"
    destino.unlink(missing_ok=True)

    try:
        criar_stage(con, arquivo, paralelo=True)
        metricas = con.execute(SQL_VALIDACAO.replace('{ANO}', str(ano))).fetchone()
        con.execute(
            f"COPY ({SQL_TRATAR}) TO '{destino}' "
            f"(FORMAT PARQUET, COMPRESSION 'ZSTD')"
        )
    except duckdb.NotImplementedException:
        print(f"  [{ano}] reader paralelo recusou o arquivo; retry serial", flush=True)
        criar_stage(con, arquivo, paralelo=False)
        destino.unlink(missing_ok=True)
        metricas = con.execute(SQL_VALIDACAO.replace('{ANO}', str(ano))).fetchone()
        con.execute(
            f"COPY ({SQL_TRATAR}) TO '{destino}' "
            f"(FORMAT PARQUET, COMPRESSION 'ZSTD')"
        )

    con.execute("DROP TABLE IF EXISTS stage")
    linhas = con.execute(f"SELECT count(*) FROM read_parquet('{destino}')").fetchone()[0]
    if linhas != metricas[0]:
        raise AssertionError(f"[{ano}] contagem divergente: fonte={metricas[0]} parquet={linhas}")

    nomes = [
        "linhas", "data_ruim", "ano_divergente", "hora_ruima",
        "uf_placa_invalida", "uf_infracao_invalida", "km_invalido",
        "qtd_invalida", "br_invalido", "especie_sentinela",
    ]
    m = dict(zip(nomes, metricas))
    m["ano"] = ano
    m["parquet_mb"] = round(destino.stat().st_size / 1e6, 1)
    return m


# ---------------------------------------------------------------------------
# fase B — resumo das colunas
# ---------------------------------------------------------------------------
def fase_b(con: duckdb.DuckDBPyConnection, validacoes: list[dict]) -> Path:
    glob = str(BASE_DIR / "*.parquet")
    con.execute(f"CREATE OR REPLACE VIEW base AS SELECT * FROM read_parquet('{glob}')")
    total = con.execute("SELECT count(*) FROM base").fetchone()[0]
    if total != TOTAL_ESPERADO:
        raise AssertionError(f"TOTAL global {total:,} != esperado {TOTAL_ESPERADO:,}")

    cols = [
        r[0] for r in con.execute(
            "SELECT column_name FROM information_schema.columns "
            "WHERE table_name='base' ORDER BY ordinal_position"
        ).fetchall()
    ]
    tipos = dict(con.execute(
        "SELECT column_name, data_type FROM information_schema.columns "
        "WHERE table_name='base'"
    ).fetchall())

    aggs = ["count(*) AS n"]
    for c in cols:
        aggs.append(f"count(*) FILTER (WHERE \"{c}\" IS NULL) AS \"nul_{c}\"")
        aggs.append(f"approx_count_distinct(\"{c}\") AS \"d_{c}\"")
    row = dict(zip(
        ["n"] + [x for c in cols for x in (f"nul_{c}", f"d_{c}")],
        con.execute(f"SELECT {', '.join(aggs)} FROM base").fetchone(),
    ))

    # nulos por era (era1 2007-18 | era2 2019-21 | era3 2022-25)
    eras = {"era1": "ANO BETWEEN 2007 AND 2018", "era2": "ANO BETWEEN 2019 AND 2021",
            "era3": "ANO >= 2022"}
    aggs_e = []
    for c in cols:
        for e, w in eras.items():
            aggs_e.append(
                f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL AND {w}) "
                f"/ nullif(count(*) FILTER (WHERE {w}), 0), 1) AS \"{e}__{c}\""
            )
    row_e = dict(zip(
        [f"{e}__{c}" for c in cols for e in eras],
        con.execute(f"SELECT {', '.join(aggs_e)} FROM base").fetchone(),
    ))

    # redundancias / resumos possiveis
    red = con.execute("""
        SELECT
            approx_count_distinct(COD_INFRACAO) AS n_cod,
            approx_count_distinct(COD_INFRACAO || '|' || coalesce(ENQUADRAMENTO,'')) AS n_cod_enq,
            approx_count_distinct(COD_INFRACAO || '|' || coalesce(DESCRICAO_ABRVIADA,'')) AS n_cod_desc,
            approx_count_distinct(ENQUADRAMENTO || '|' || coalesce(DESCRICAO_ABRVIADA,'')) AS n_enq_desc,
            approx_count_distinct(NOME_VEICULO_MARCA) AS n_marca,
            approx_count_distinct(NOM_MODELO_VEICULO) AS n_modelo,
            approx_count_distinct(NOM_MUNICIPIO) AS n_municipio,
            count(*) FILTER (WHERE MED_REALIZADA IS NOT NULL
                 AND try_cast(replace(MED_REALIZADA,',','.') AS DOUBLE) IS NULL) AS med_ruim,
            count(*) FILTER (WHERE MED_REALIZADA IS NOT NULL) AS med_nao_nulo,
            count(*) FILTER (WHERE MED_CONSIDERADA IS NOT NULL
                 AND try_cast(replace(MED_CONSIDERADA,',','.') AS DOUBLE) IS NULL) AS medc_ruim,
            count(*) FILTER (WHERE MED_CONSIDERADA IS NOT NULL) AS medc_nao_nulo
        FROM base
    """).fetchone()
    red = dict(zip(
        ["n_cod", "n_cod_enq", "n_cod_desc", "n_enq_desc", "n_marca", "n_modelo",
         "n_municipio", "med_ruim", "med_nao_nulo", "medc_ruim", "medc_nao_nulo"], red))

    # amostras de MED_*
    amostras = con.execute("""
        SELECT MED_REALIZADA, count(*) FROM base
        WHERE MED_REALIZADA IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 6
    """).fetchall()

    # ------------------------------------------------------------------ md
    def pct(c: str) -> str:
        return f"{100.0 * row[f'nul_{c}'] / row['n']:.2f}%"

    linhas_md = [
        "# resumo_colunas — base analítica (dados_tratados)",
        "",
        f"- **Linhas:** {row['n']:,} · **Colunas:** {len(cols)} · "
        f"Gerado por `scripts/04_tratar.py` em {time.strftime('%Y-%m-%d %H:%M')}.",
        "- **Fonte:** `dados_brutos/infracoes_2007..2025.csv.gz` (19 arquivos, schema canônico 25 col).",
        "- Colunas de texto: sentinela → `NULL`; `TRIM` + `MAIÚSCULAS` + acentos→ASCII. "
        "Colunas derivadas: `MES` (nome do mês: JANEIRO…DEZEMBRO), `ANO`, `QUARTIL` (Q1=jan-mar, Q2=abr-jun, Q3=jul-set, Q4=out-dez).",
        "",
        "## 1. Estatísticas globais",
        "",
        "| # | Coluna | Tipo | Nulos | Nulos % | Distintos (aprox) |",
        "|---|--------|------|------:|--------:|------------------:|",
    ]
    for i, c in enumerate(cols, 1):
        linhas_md.append(
            f"| {i} | `{c}` | {tipos[c]} | {row[f'nul_{c}']:,} | {pct(c)} | {row[f'd_{c}']:,} |"
        )

    linhas_md += [
        "",
        "## 2. Nulos por era (era1 2007–2018 · era2 2019–2021 · era3 2022–2025)",
        "",
        "| Coluna | era1 | era2 | era3 |",
        "|--------|-----:|-----:|-----:|",
    ]
    for c in cols:
        v = [row_e[f"{e}__{c}"] for e in eras]
        linhas_md.append(f"| `{c}` | {v[0]}% | {v[1]}% | {v[2]}% |")

    linhas_md += ["", "## 3. Anomalias corrigidas na padronização", "",
                  "| Ano | Linhas | Datas inválidas | Ano divergente | Hora inválida | UF placa inv. | UF infra inv. | KM inv. | Qtd inv. | BR inv. | Espécie sentinela | Parquet MB |",
                  "|----:|-------:|----------------:|---------------:|--------------:|--------------:|--------------:|--------:|---------:|--------:|------------------:|-----------:|"]
    for m in validacoes:
        linhas_md.append(
            f"| {m['ano']} | {m['linhas']:,} | {m['data_ruim']} | {m['ano_divergente']} | "
            f"{m['hora_ruima']} | {m['uf_placa_invalida']} | {m['uf_infracao_invalida']} | "
            f"{m['km_invalido']} | {m['qtd_invalida']} | {m['br_invalido']} | "
            f"{m['especie_sentinela']} | {m['parquet_mb']} |"
        )

    # ---------------------------------------------------------- recomendacoes
    fd_enq = red["n_cod_enq"] <= red["n_cod"] * 1.02
    fd_desc = red["n_cod_desc"] <= red["n_cod"] * 1.02
    med_ok = red["med_nao_nulo"] > 0 and red["med_ruim"] == 0
    medc_ok = red["medc_nao_nulo"] > 0 and red["medc_ruim"] == 0

    linhas_md += [
        "",
        "## 4. Colunas que PODEM ser resumidas",
        "",
        f"- **`DESCRICAO_ABRVIADA` / `ENQUADRAMENTO` vs `COD_INFRACAO`**: "
        f"{red['n_cod']:,} códigos distintos; pares (código,enquadramento) ≈ {red['n_cod_enq']:,}; "
        f"pares (código,descrição) ≈ {red['n_cod_desc']:,}; "
        f"(enquadramento,descrição) ≈ {red['n_enq_desc']:,}. "
        + ("Os textos são **funcionalmente dependentes do código** → em bases "
           "modeláveis podem ser **resumidos em `COD_INFRACAO`** (dicionário à parte), "
           "eliminando redundância." if fd_enq and fd_desc else
           "Há códigos com múltiplos textos → manter os três na base analítica; "
           "na modelável resumir via dicionário código→texto."),
        f"- **`NOME_VEICULO_MARCA` + `NOM_MODELO_VEICULO`**: ≈ {red['n_marca']:,} marcas e "
        f"≈ {red['n_modelo']:,} modelos (carteira aberta, grafias livres) → **resumir em "
        "`ESPECIE`/`TIPO_VEICULO`** (já padronizados) ou dicionário de marcas controlado "
        "antes de uso como categoria.",
        f"- **`NOM_MUNICIPIO`**: ≈ {red['n_municipio']:,} municípios distintos (texto) → "
        "**resumir em código IBGE** (ou `UF_INFRACAO`+código) para agregação/geolocalização "
        "estável.",
        "- **`HORA`**: detalhe fino já coberto por `MES`/`ANO`/`QUARTIL` → em bases "
        "modeláveis **resumir em turno/faixa horária** (MADRUGADA/MANHÃ/TARDE/NOITE).",
        "- **`NUMERO_AUTO`**: identificador (≈1 por linha) — **não resume**: é a chave do auto "
        "de infração; servir de join/contagem, não de categoria.",
        "",
        "## 5. Colunas que PRECISAM de resumo/atenção",
        "",
        "- **Lacunas de era** (null100% em eras inteiras — ausência legítima na fonte, "
        "não erro): `NUMERO_AUTO` (era1), `HORA` (era1–2 parcial), `TIPO_VEICULO` (era1–2), "
        "`QTD_INFRACOES` (era1–2), `COD_INFRACAO` (2007) — ver matriz §2. "
        "Em cruzamentos entre eras, usar apenas colunas com cobertura nas duas.",
        "- **Sentinelas de origem** (`'-1`, `-1`, `NA`…) convertidas para `NULL` "
        "(§3) — nunca imputar 0/'' nelas: o `NULL` é a marca de \"não informado\".",
        f"- **`MED_REALIZADA`/`MED_CONSIDERADA`**: "
        + (f"100% numéricas após vírgula→ponto (0 inválidas em "
           f"{red['med_nao_nulo']:,}/{red['medc_nao_nulo']:,}) → **podem virar DOUBLE** "
           "numa próxima etapa sem perda." if med_ok and medc_ok else
           "ainda com valores não numéricos misturados → **precisam de resumo/conversão** "
           "controlada (mapear unidades) antes de virar DOUBLE."),
        f"- **`UF_PLACA`/`UF_INFRACAO`**: apenas 27 UFs aceitas; inválidas viram `NULL` "
        f"({sum(m['uf_placa_invalida'] + m['uf_infracao_invalida'] for m in validacoes):,} "
        "células corrigidas no total).",
        "",
        "---",
        "Leitura: `read_parquet('dados_tratados/base_analitica/*.parquet')` (DuckDB/pandas).",
    ]

    out = TRATADOS / "resumo_colunas.md"
    out.write_text("\n".join(linhas_md) + "\n", encoding="utf-8")
    print(f"resumo -> {out}", flush=True)
    return out


def main() -> None:
    argv = sys.argv[1:]
    so_resumo = "--resumo" in argv
    anos = [int(a) for a in argv if a.isdigit()] or ([] if so_resumo else list(range(2007, 2026)))
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    if len(anos) == 19:
        VALIDACAO.unlink(missing_ok=True)  # recomeco completo

    con = duckdb.connect()
    con.execute("SET threads=2")
    con.execute("SET memory_limit='2400MB'")
    con.execute(f"SET temp_directory='{TMP_DIR}'")
    con.execute("SET preserve_insertion_order=false")

    validacoes: list[dict] = []
    t0 = time.time()
    for ano in anos:
        ta = time.time()
        m = tratar_ano(con, ano)
        validacoes.append(m)
        with VALIDACAO.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(m, ensure_ascii=False) + "\n")
        print(
            f"[{ano}] {m['linhas']:,} linhas -> {m['parquet_mb']} MB "
            f"({time.time() - ta:.0f}s) | uf_placa_inv={m['uf_placa_invalida']} "
            f"uf_infra_inv={m['uf_infracao_invalida']} especie_sent={m['especie_sentinela']} "
            f"data_ruim={m['data_ruim']} ano_div={m['ano_divergente']}",
            flush=True,
        )

    if so_resumo or len(anos) == 19:
        if so_resumo:
            validacoes = [
                json.loads(l)
                for l in VALIDACAO.read_text(encoding="utf-8").splitlines()
                if l.strip()
            ]
        # fase B so faz sentido com a base completa (19 no jsonl)
        fase_b(con, validacoes)
    else:
        print("parcial: pulando fase B (rode com --resumo apos os 19)", flush=True)

    if so_resumo:
        print("resumo ok", flush=True)
        return
    total = sum(m["linhas"] for m in validacoes)
    print(f"OK {len(anos)} anos, {total:,} linhas, {time.time() - t0:.0f}s", flush=True)
    if len(anos) == 19 and total != TOTAL_ESPERADO:
        raise SystemExit(f"TOTAL {total} != {TOTAL_ESPERADO}")


if __name__ == "__main__":
    main()

Overwriting scripts/04_tratar.py


**Reexecução (opcional):** a Fase 1 hoje **parte dos `.csv.gz` de
`dados_brutos/`** (`01_ingestao_brutos.py` — sem Google Drive); a Fase 2 roda
direto sobre os mesmos `.csv.gz` — cerca de 1 h:


In [6]:
if EXECUTAR_PIPELINE:
    import subprocess, sys
    subprocess.run([sys.executable, "scripts/04_tratar.py"], check=True)
else:
    print("EXECUTAR_PIPELINE=False — pulando reexecução; resultados já em dados_tratados/")

EXECUTAR_PIPELINE=False — pulando reexecução; resultados já em dados_tratados/


## 4. Validações executáveis

Rodem em segundos sobre os parquets gerados (se `dados_tratados/` não existir, rode a Fase 2 primeiro).

### 4.1 Contagem total e período

In [7]:
r = con.execute("""
SELECT count(*) total, count(DISTINCT ANO) anos,
       min(DATA_INFRACAO) dmin, max(DATA_INFRACAO) dmax
FROM read_parquet('dados_tratados/base_analitica/*.parquet')
""").fetchone()
print(f"Linhas: {r[0]:,} | Anos: {r[1]} | Periodo: {r[2]} a {r[3]}")
assert r[0] == 92_215_588, "total divergente"
print("OK: 92.215.588 linhas")

Linhas: 92,215,588 | Anos: 19 | Periodo: 2007-01-01 a 2025-12-31
OK: 92.215.588 linhas


### 4.2 `MES` (nome) e `QUARTIL` (4×3 meses) coerentes com `DATA_INFRACAO`

In [8]:
meses = [x[0] for x in con.execute("""
    SELECT MES FROM read_parquet('dados_tratados/base_analitica/*.parquet')
    GROUP BY MES ORDER BY MIN(DATA_INFRACAO)
""").fetchall()]
print("MES:", meses)

r = con.execute("""
SELECT count(*),
       count(DISTINCT MES),
       count(*) FILTER (WHERE MES IS NULL),
       count(*) FILTER (WHERE MES <> CASE EXTRACT(MONTH FROM DATA_INFRACAO)
            WHEN 1 THEN 'JANEIRO' WHEN 2 THEN 'FEVEREIRO' WHEN 3 THEN 'MARCO'
            WHEN 4 THEN 'ABRIL' WHEN 5 THEN 'MAIO' WHEN 6 THEN 'JUNHO'
            WHEN 7 THEN 'JULHO' WHEN 8 THEN 'AGOSTO' WHEN 9 THEN 'SETEMBRO'
            WHEN 10 THEN 'OUTUBRO' WHEN 11 THEN 'NOVEMBRO' WHEN 12 THEN 'DEZEMBRO' END),
       count(*) FILTER (WHERE QUARTIL <> CASE
            WHEN EXTRACT(MONTH FROM DATA_INFRACAO) <= 3 THEN 'Q1'
            WHEN EXTRACT(MONTH FROM DATA_INFRACAO) <= 6 THEN 'Q2'
            WHEN EXTRACT(MONTH FROM DATA_INFRACAO) <= 9 THEN 'Q3' ELSE 'Q4' END)
FROM read_parquet('dados_tratados/base_analitica/*.parquet')
""").fetchone()
print(f"total={r[0]:,} distinct_MES={r[1]} nulos_MES={r[2]} MES_fora={r[3]} QUARTIL_div={r[4]}")
assert r[1] == 12 and r[2] == 0 and r[3] == 0 and r[4] == 0
print("OK: MES = 12 nomes, QUARTIL = Q1..Q4, tudo coerente")

MES: ['JANEIRO', 'FEVEREIRO', 'MARCO', 'ABRIL', 'MAIO', 'JUNHO', 'JULHO', 'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']


total=92,215,588 distinct_MES=12 nulos_MES=0 MES_fora=0 QUARTIL_div=0
OK: MES = 12 nomes, QUARTIL = Q1..Q4, tudo coerente


### 4.3 Schema: 37 colunas, todas em MAIÚSCULOS

In [9]:
cols = con.execute("""
    DESCRIBE SELECT * FROM read_parquet('dados_tratados/base_analitica/infracoes_2007.parquet')
""").fetchall()
print(f"{len(cols)} colunas | todas MAIUSCULAS: {all(c[0] == c[0].upper() for c in cols)}")
for c in cols:
    print(f"  {c[0]:<24} {c[1]}")
assert len(cols) == 37 and all(c[0] == c[0].upper() for c in cols)

37 colunas | todas MAIUSCULAS: True
  NUMERO_AUTO              VARCHAR
  DATA_INFRACAO            DATE
  HORA                     TIME
  TIP_ABORDAGEM            VARCHAR
  IND_ASSINOU_AUTO         VARCHAR
  IND_VEICULO_ESTRANGEIRO  VARCHAR
  SENTIDO_TRAFEGO          VARCHAR
  UF_PLACA                 VARCHAR
  UF_INFRACAO              VARCHAR
  NUM_BR_INFRACAO          BIGINT
  NUM_KM_INFRACAO          DOUBLE
  NOM_MUNICIPIO            VARCHAR
  COD_INFRACAO             VARCHAR
  DESCRICAO_ABRVIADA       VARCHAR
  ENQUADRAMENTO            VARCHAR
  DATA_INICIO_VIGENCIA     DATE
  DATA_FIM_VIGENCIA        DATE
  MED_REALIZADA            VARCHAR
  MED_CONSIDERADA          VARCHAR
  EXC_VERIFICADO           VARCHAR
  ESPECIE                  VARCHAR
  NOME_VEICULO_MARCA       VARCHAR
  NOM_MODELO_VEICULO       VARCHAR
  TIPO_VEICULO             VARCHAR
  QTD_INFRACOES            BIGINT
  MES                      VARCHAR
  ANO                      SMALLINT
  QUARTIL                  VARCHA

### 4.4 Amostra: era1 (2010, lacunas legítimas) vs era3 (2024, completo)

In [10]:
for ano in (2010, 2024):
    row = con.execute(f"""
        SELECT DATA_INFRACAO, MES, ANO, QUARTIL, HORA, UF_INFRACAO,
               ESPECIE, TIPO_VEICULO, QTD_INFRACOES
        FROM read_parquet('dados_tratados/base_analitica/*.parquet')
        WHERE ANO = {ano} LIMIT 2
    """).fetchall()
    print(f"--- {ano} ---")
    for x in row:
        print("  ", x)

--- 2010 ---
   (datetime.date(2010, 1, 30), 'JANEIRO', 2010, 'Q1', None, 'MG', 'MISTO', None, None)
   (datetime.date(2010, 1, 30), 'JANEIRO', 2010, 'Q1', None, 'PR', 'CARGA', None, None)
--- 2024 ---
   (datetime.date(2024, 1, 2), 'JANEIRO', 2024, 'Q1', datetime.time(2, 0), 'ES', 'MISTO', 'UTILITARIO', 1)
   (datetime.date(2024, 1, 2), 'JANEIRO', 2024, 'Q1', datetime.time(2, 0), 'ES', 'PASSAGEIRO', 'AUTOMOVEL', 1)


### 4.5 Nulos globais (resumo rápido das colunas)

In [11]:
cols = [c[0] for c in con.execute("""
    DESCRIBE SELECT * FROM read_parquet('dados_tratados/base_analitica/infracoes_2007.parquet')
""").fetchall()]
aggs = ", ".join(
    f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL) / count(*), 1) AS \"{c}\""
    for c in cols
)
row = con.execute(
    f"SELECT {aggs} FROM read_parquet('dados_tratados/base_analitica/*.parquet')"
).fetchone()
for c, v in sorted(zip(cols, row), key=lambda x: -x[1]):
    if v and v > 0:
        print(f"  {c:<24} {v}%")
print("(so colunas com nulos > 0; ver resumo_colunas.md para a matriz por era)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  TIPO_VEICULO             68.6%
  DATA_FIM_VIGENCIA        64.8%
  QTD_INFRACOES            57.3%
  IND_ASSINOU_AUTO         56.5%
  NUMERO_AUTO              51.0%
  COD_INFRACAO             36.5%
  NOME_VEICULO_MARCA       31.8%
  MED_REALIZADA            29.6%
  HORA                     24.6%
  TURNO                    24.6%
  FAIXA_HORARIA            24.6%
  SENTIDO_TRAFEGO          23.7%
  NOM_MODELO_VEICULO       20.3%
  EXC_VERIFICADO           19.8%
  MED_CONSIDERADA          19.2%
  ESPECIE                  18.6%
  UF_PLACA                 15.6%
  DATA_INICIO_VIGENCIA     0.1%
(so colunas com nulos > 0; ver resumo_colunas.md para a matriz por era)


### 4.6 Relatórios gerados na Fase 2

In [12]:
for nome in ("dados_tratados/resumo_colunas.md", "dados_tratados/valores_rejeitados.md"):
    p = Path(nome)
    print("=" * 30, nome, "=" * 30)
    if p.exists():
        linhas = p.read_text(encoding="utf-8").splitlines()
        print("\n".join(linhas[:25]))
        print(f"... ({len(linhas)} linhas no total)")
    else:
        print("AUSENTE — rode: .venv/bin/python scripts/04_tratar.py")
    print()

============================== dados_tratados/resumo_colunas.md ==============================
# resumo_colunas — base analítica (dados_tratados)

- **Linhas:** 92,215,588 · **Colunas:** 28 · Gerado por `scripts/04_tratar.py` em 2026-11-26 00:29.
- **Fonte:** `dados_brutos/infracoes_2007..2025.csv.gz` (19 arquivos, schema canônico 25 col).
- Colunas de texto: sentinela → `NULL`; `TRIM` + `MAIÚSCULAS` + acentos→ASCII. Colunas derivadas: `MES` (nome do mês: JANEIRO…DEZEMBRO), `ANO`, `QUARTIL` (Q1=jan-mar, Q2=abr-jun, Q3=jul-set, Q4=out-dez).

## 1. Estatísticas globais

| # | Coluna | Tipo | Nulos | Nulos % | Distintos (aprox) |
|---|--------|------|------:|--------:|------------------:|
| 1 | `NUMERO_AUTO` | VARCHAR | 47,053,140 | 51.03% | 46,181,829 |
| 2 | `DATA_INFRACAO` | DATE | 0 | 0.00% | 6,969 |
| 3 | `HORA` | TIME | 22,679,289 | 24.59% | 85,475 |
| 4 | `TIP_ABORDAGEM` | VARCHAR | 0 | 0.00% | 2 |
| 5 | `IND_ASSINOU_AUTO` | VARCHAR | 52,097,346 | 56.50% | 2 |
| 6 | `IND_VEICULO_ES

## 5. Histórico de alterações

| # | Alteração | Onde |
|---|-----------|------|
| 1 | Fase 1 completa: RARs → unificação → normalização 25 col (92.215.588) | `extract_rar.js`, `01`, `02`, `03` · PR #1 |
| 2 | **jun/2024 inexistente** → `MESES_INEXISTENTES = {2024: [6]}` (INFO, não pendência) | `02` · commit `5c3caba` |
| 3 | Publicação de `dados_brutos/` no GitHub: gzip + volumes ≤95 MB (limite Git 100 MB) + `SHA256SUMS.txt` (uploads de Release bloqueados no sandbox) | PR #2 |
| 4 | Fallback `parallel=false` no `03` (reader do DuckDB recusa o 2023) | `03` |
| 5 | **Fase 2**: `04_tratar` → base analítica **28 colunas** Parquet; sentinela/UF/acentos padronizados | PR #3 |
| 6 | **`QUARTIL` = 4 períodos de 3 meses** (Q1–Q4) confirmada pelo usuário | `04` |
| 7 | **`MES`: número → nome do mês** (JANEIRO…DEZEMBRO) — migração dos 19 parquets em 74 s | `04` · commit `ff7746c` |
| 8 | **`valores_rejeitados.md`**: lista completa do que virou `NULL` (108 valores de UF; 2 sentinelas de `ESPECIE`) | varredura da fonte |
| 9 | **Ingestão reancorada em `dados_brutos/*.csv.gz`** — novo `01_ingestao_brutos.py` (contagens + header + `--sha256`); scripts de Drive (`extract_rar`, `01`, `01b`, `02`) removidos | varredura · PR #10 |

## Referências

- `dados_brutos/README.md` — 19 CSVs, checksums, regeneração (Release [`dados-brutos`](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/releases/tag/dados-brutos))
- `dados_tratados/README.md` · `resumo_colunas.md` · `valores_rejeitados.md`
- `logs/ingestao_multas.md` — log completo das execuções
- PRs: [#1](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/pull/1) · [#2](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/pull/2) · [#3](https://github.com/Matheus-Trajano/PRF-MULTAS-PROJETO/pull/3)